# 7 正则化

## 学习目标

* 知道正则化原因和方法

在训练深层神经网络时，由于模型参数较多，在数据量不足的情况下，很容易过拟合。Dropout 就是在神经网络中一种缓解过拟合的方法。

## 1. Dropout 层的原理和使用

我们知道，缓解过拟合的方式就是降低模型的复杂度，而 Dropout 就是通过减少神经元之间的连接，把稠密的神经网络神经元连接，变成稀疏的神经元连接，从而达到降低网络复杂度的目的。

我们先通过一段代码观察下丢弃层的效果：

In [ ]:
import torch
import torch.nn as nn

def test():

    # 初始化丢弃层
    dropout = nn.Dropout(p=0.8)
    # 初始化输入数据
    inputs = torch.randint(0, 10, size=[5, 8]).float()
    print(inputs)
    print('-' * 50)

    outputs = dropout(inputs)
    print(outputs)

if __name__ == '__main__':
    test()

tensor([[5., 7., 3., 6., 5., 7., 6., 6.],
        [6., 4., 4., 1., 8., 0., 4., 1.],
        [2., 3., 1., 7., 4., 6., 6., 5.],
        [9., 5., 5., 0., 0., 4., 2., 3.],
        [9., 2., 7., 0., 5., 7., 7., 1.]])
--------------------------------------------------
tensor([[ 0., 35.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  5.,  0.,  0., 20.,  5.],
        [10.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [45.,  0., 25.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0., 35.,  0.]])


我们将 Dropout 层的概率 p 设置为 0.8，此时经过 Dropout 层计算的张量中就出现了很多 0，概率 p 设置值越大，则张量中出现的 0 就越多。上面结果的计算过程如下：

1. 先按照 p 设置的概率，随机将部分的张量元素设置为 0
2. 为了校正张量元素被设置为 0 带来的影响，需要对非 0 的元素进行缩放，其缩放因子为：1/(1-p)，上面代码中 p 的值为 0.8，根据公式缩放因子为：1/(1-0.8) = 5
3. 比如：第 3 个元素，原来是 3，乘以缩放因子之后变成 15。

我们也发现了，丢弃概率 p 的值越大，则缩放因子的值就越大，相对其他未被设置的元素就要更多的变大。丢弃概率 P 的值越小，则缩放因子的值就越小，相对应其他未被设置为 0 的元素就要有较小的变大。

当张量某些元素被设置为 0 时，对网络会带来什么影响?

比如上面这种情况，如果输入该样本，会使得某些参数无法更新，请看下面的代码：

In [2]:
import torch
import torch.nn as nn

# 设置随机数种子
torch.manual_seed(0)


def caculate_gradient(x, w):

    y = x @ w
    y = y.sum()
    y.backward()
    print('Gradient:', w.grad.reshape(1, -1).squeeze().numpy())


def test01():

    # 初始化权重
    w = torch.randn(15, 1, requires_grad=True)
    # 初始化输入数据
    x = torch.randint(0, 10, size=[5, 15]).float()
    # 计算梯度
    caculate_gradient(x, w)


def test02():

    # 初始化权重
    w = torch.randn(15, 1, requires_grad=True)
    # 初始化输入数据
    x = torch.randint(0, 10, size=[5, 15]).float()
    # 初始化丢弃层
    dropout = nn.Dropout(p=0.8)
    x = dropout(x)
    # 计算梯度
    caculate_gradient(x, w)
    
if __name__ == '__main__':
    test01()
    print('-' * 70)
    test02()

Gradient: [19. 15. 16. 13. 34. 23. 20. 22. 23. 26. 21. 29. 28. 22. 29.]
----------------------------------------------------------------------
Gradient: [20.  0. 10. 10. 20. 70.  0.  0.  0. 20. 20. 30. 10.  0.  0.]


从程序结果来看，是否经过 Dropout 层对梯度的计算产生了不小的影响，例如：经过 Dropout 层之后有一些梯度为 0，这使得参数无法得到更新，从而达到了降低网络复杂度的目的。

## 2. 小节
在本小节中，我们主要学习了 dropout 层的使用，其作用用于控制网络复杂度，达到正则化的目的，类似于 L2 正则化对线性回归的作用。所以，同学们以后经常会看到该层在网络中出现。